In [ ]:
# @title 1) Student Info & Config
STUDENT_NAME = "Воропаев Фёдор Игоревич"  # @param {type:"string"}
GROUP = "11-301"         # @param {type:"string"}
ASSIGMENT = "CV_REAL_WORLD_PRACTICE"
SEED = 42
START_DATE = "2026-04-07"
DUE_DATE = "2026-04-14"


# HW5 — Классификация + Transfer Learning (PyTorch)

**Тема:** pretrained backbone, head, freeze/unfreeze, discriminative LR, базовые метрики.

Фокус: правильно собрать pipeline и показать понимание transfer learning.
Тесты используют **FakeData**, чтобы проверка была быстрой и не зависела от интернета.

Баллы: **100**


In [ ]:
# Install deps
!pip -q install -U torch torchvision torchaudio scikit-learn matplotlib


In [ ]:
import random
from datetime import datetime, timezone
from typing import Dict, Tuple, Any

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import accuracy_score, f1_score

plt.rcParams["figure.dpi"] = 140

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---------------- Reproducibility ----------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ---------------- Dates for last cell ----------------
def _parse_date(s: str):
    try:
        return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def _sec(td) -> float:
    return float(td.total_seconds())

start_dt = _parse_date(START_DATE)
due_dt = _parse_date(DUE_DATE)
submission_dt = datetime.now(timezone.utc)

# ---------------- Scoring ----------------
SCORES: Dict[str, float] = {}

def _set_score(task: str, pts: float):
    SCORES[task] = float(pts)

def _total_score() -> float:
    return float(sum(SCORES.values()))

def _print_scores():
    print("Scores:")
    for k in sorted(SCORES.keys()):
        print(f"  {k}: {SCORES[k]:.1f}")
    print("TOTAL:", _total_score())

def _assert(cond: bool, msg: str):
    if not cond:
        raise AssertionError(msg)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


## Данные для тестов (FakeData)

Тесты не требуют интернета и не скачивают датасеты.
FakeData генерирует синтетические изображения нужного размера.


In [ ]:
def get_fake_loaders(batch_size=32, seed=123, train_tf=None, eval_tf=None):
    seed_everything(seed)
    train_ds = datasets.FakeData(size=256, image_size=(3,224,224), num_classes=10, transform=train_tf)
    dev_ds   = datasets.FakeData(size=128, image_size=(3,224,224), num_classes=10, transform=eval_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    dev_loader   = DataLoader(dev_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, dev_loader


## Task 1 (15 pts) — Transforms под pretrained

Реализуйте `get_transforms()` и верните `(train_tf, eval_tf)`.

Требования:
- `train_tf`: Resize(256) + RandomResizedCrop(224) + RandomHorizontalFlip + ColorJitter + ToTensor + Normalize(ImageNet)
- `eval_tf`: Resize(256) + CenterCrop(224) + ToTensor + Normalize(ImageNet), **без рандома**


In [ ]:
# TODO: implement
def get_transforms():
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 1
try:
    train_tf, eval_tf = get_transforms()
    _assert(isinstance(train_tf, transforms.Compose), "train_tf must be transforms.Compose")
    _assert(isinstance(eval_tf, transforms.Compose), "eval_tf must be transforms.Compose")

    names_train = [t.__class__.__name__ for t in train_tf.transforms]
    names_eval  = [t.__class__.__name__ for t in eval_tf.transforms]

    required_train = {"Resize","RandomResizedCrop","RandomHorizontalFlip","ColorJitter","ToTensor","Normalize"}
    _assert(required_train.issubset(set(names_train)), f"train_tf missing: {required_train - set(names_train)}")

    required_eval = {"Resize","CenterCrop","ToTensor","Normalize"}
    _assert(required_eval.issubset(set(names_eval)), f"eval_tf missing: {required_eval - set(names_eval)}")

    from PIL import Image
    img = Image.fromarray((np.random.rand(256,256,3)*255).astype(np.uint8))
    a = eval_tf(img)
    b = eval_tf(img)
    _assert(torch.allclose(a, b), "eval_tf must be deterministic")
    _set_score("task1", 15)
    print("✅ Task 1 passed")
except Exception as e:
    print("❌ Task 1 failed:", e)
    _set_score("task1", 0)


## Task 2 (15 pts) — Модель: ResNet18 pretrained + новый head

Реализуйте `build_model(num_classes)`:
- загрузите pretrained ResNet18
- замените `fc` на `Linear(..., num_classes)`


In [ ]:
# TODO: implement
def build_model(num_classes: int) -> nn.Module:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 2
try:
    m = build_model(10)
    _assert(isinstance(m, nn.Module), "Must return nn.Module")
    _assert(isinstance(m.fc, nn.Linear), ".fc must be nn.Linear after replacement")
    _assert(m.fc.out_features == 10, "Head must output num_classes")
    _set_score("task2", 15)
    print("✅ Task 2 passed")
except Exception as e:
    print("❌ Task 2 failed:", e)
    _set_score("task2", 0)


## Task 3 (10 pts) — Freeze backbone

Реализуйте `freeze_backbone(model)`:
- заморозить все параметры backbone
- head (`fc`) оставить обучаемым


In [ ]:
# TODO: implement
def freeze_backbone(model: nn.Module) -> None:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 3
try:
    m = build_model(10)
    freeze_backbone(m)
    back_ok = all((not p.requires_grad) for n,p in m.named_parameters() if not n.startswith("fc."))
    head_ok = all((p.requires_grad) for n,p in m.named_parameters() if n.startswith("fc."))
    _assert(back_ok, "Backbone params must be frozen")
    _assert(head_ok, "Head params must be trainable")
    _set_score("task3", 10)
    print("✅ Task 3 passed")
except Exception as e:
    print("❌ Task 3 failed:", e)
    _set_score("task3", 0)


## Task 4 (15 pts) — Optimizer: discriminative LR

Реализуйте `make_optimizer(model, lr_backbone, lr_head)`:
- используйте AdamW
- две группы параметров: backbone и head


In [ ]:
# TODO: implement
def make_optimizer(model: nn.Module, lr_backbone: float = 1e-4, lr_head: float = 3e-4) -> torch.optim.Optimizer:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 4
try:
    m = build_model(10)
    freeze_backbone(m)
    opt = make_optimizer(m, 1e-4, 3e-4)
    _assert(len(opt.param_groups) == 2, "Expected 2 param groups")
    lrs = sorted([pg["lr"] for pg in opt.param_groups])
    _assert(abs(lrs[0]-1e-4) < 1e-12 and abs(lrs[1]-3e-4) < 1e-12, "Incorrect LRs in param groups")
    _set_score("task4", 15)
    print("✅ Task 4 passed")
except Exception as e:
    print("❌ Task 4 failed:", e)
    _set_score("task4", 0)


## Task 5 (15 pts) — Train one epoch

Реализуйте `train_one_epoch(model, loader, optimizer)`:
- `.train()`
- forward → cross_entropy → backward → step
- вернуть средний loss


In [ ]:
# TODO: implement
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 5
try:
    train_tf, eval_tf = get_transforms()
    tr_loader, _ = get_fake_loaders(batch_size=32, seed=123, train_tf=train_tf, eval_tf=eval_tf)

    m = build_model(10).to(DEVICE)
    freeze_backbone(m)
    opt = make_optimizer(m, 1e-4, 3e-4)

    loss = train_one_epoch(m, tr_loader, opt)
    _assert(isinstance(loss, (float, np.floating)), "Loss must be float")
    _assert(loss > 0, "Loss must be > 0")
    _set_score("task5", 15)
    print("✅ Task 5 passed")
except Exception as e:
    print("❌ Task 5 failed:", e)
    _set_score("task5", 0)


## Task 6 (15 pts) — Evaluate: accuracy и macro-F1

Реализуйте `evaluate(model, loader)` и верните:
- `accuracy` — доля правильных ответов
- `macro_f1` — F1, усреднённый по классам


In [ ]:
# TODO: implement
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 6
try:
    train_tf, eval_tf = get_transforms()
    _, dv_loader = get_fake_loaders(batch_size=32, seed=321, train_tf=train_tf, eval_tf=eval_tf)

    m = build_model(10).to(DEVICE)
    metrics = evaluate(m, dv_loader)
    _assert("accuracy" in metrics and "macro_f1" in metrics, "Missing keys")
    _assert(0.0 <= float(metrics["accuracy"]) <= 1.0, "accuracy in [0,1]")
    _assert(0.0 <= float(metrics["macro_f1"]) <= 1.0, "macro_f1 in [0,1]")
    _set_score("task6", 15)
    print("✅ Task 6 passed")
except Exception as e:
    print("❌ Task 6 failed:", e)
    _set_score("task6", 0)


## Task 7 (30 pts) — Pipeline: freeze vs finetune

Реализуйте `run_pipeline()`:
1) Соберите loaders (FakeData) с transforms
2) Запустите 1 эпоху для **freeze**
3) Запустите 1 эпоху для **finetune**
4) Верните словарь:
- `freeze_trainable_params`
- `finetune_trainable_params`
- `freeze_dev_accuracy`
- `finetune_dev_accuracy`

Критерий: finetune должен иметь больше обучаемых параметров.


In [ ]:
# TODO: implement
def run_pipeline() -> Dict[str, float]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 7
try:
    res = run_pipeline()
    for k in ["freeze_trainable_params","finetune_trainable_params","freeze_dev_accuracy","finetune_dev_accuracy"]:
        _assert(k in res, f"Missing key: {k}")
    _assert(float(res["freeze_trainable_params"]) > 0, "freeze_trainable_params must be > 0")
    _assert(float(res["finetune_trainable_params"]) > float(res["freeze_trainable_params"]), "finetune must have more trainable params")
    _assert(0.0 <= float(res["freeze_dev_accuracy"]) <= 1.0, "accuracy in [0,1]")
    _assert(0.0 <= float(res["finetune_dev_accuracy"]) <= 1.0, "accuracy in [0,1]")
    _set_score("task7", 30)
    print("✅ Task 7 passed")
except Exception as e:
    print("❌ Task 7 failed:", e)
    _set_score("task7", 0)


In [ ]:
# FINAL: compute total points (0..100)
total = _total_score()
_print_scores()
print("\nTOTAL POINTS / 100 =", total)


In [ ]:
def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)
	
# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
final_score = max(0.0, total * (1.0 - min(1.0, pf)))
print(final_score)
import json
final = {
    "name": STUDENT_NAME,
    "group": GROUP,
    "assignment":ASSIGMENT,
    "score": float(total),
    "penalty_score": float(final_score),
    "due_date": DUE_DATE,
    "start_date": START_DATE,
}
print(json.dumps(final, ensure_ascii=False))
